# Тестирование новых моделей
## Подготовка данных

In [1]:
import time

from typing import List

array = []

array.append("Во время обучения модели machine learning, важно обеспечить достаточное количество данных для тренировки") 
array.append("Во время обучения модели машинного обучения, важно обеспечить достаточное количество данных для тренировки")
    
array.append("Использование API позволяет интегрировать различные сервисы в ваше приложение, упрощая процесс разработки") 
array.append("The use of an API allows you to integrate various services into your application, simplifying the development process")
    
array.append("Для анализа больших объемов данных рекомендуется использовать фреймворк Hadoop, который обеспечивает эффективную обработку данных") 
array.append("В процессе дебаггинга программы была обнаружена ошибка, связанная с неправильным использованием переменных")
    
array.append("При разработке user interface необходимо учитывать принципы user experience, чтобы сделать приложение максимально удобным для пользователя") 
array.append("При разработке пользовательского интерфейса необходимо учитывать принципы пользовательского опыта, чтобы сделать приложение максимально удобным для пользователя")
    
array.append("Летний отпуск всегда запоминается яркими моментами, проведёнными на берегу моря")
array.append("Летний отпуск всегда оставляет яркие впечатления, проведённые на пляже")
    
array.append("Вчера мы с друзьями проводили время на природе, наслаждаясь пикником и свежим воздухом.")
array.append("Вчера мы с друзьями провели время в помещении, скучая и наслаждаясь теплом")
    
array.append("The latest trends in fashion emphasize sustainability and eco-friendly materials")
array.append("The latest trends in fashion ignore sustainability and promote fast fashion")
    
array.append("В этом году зима выдалась особенно холодной, и снег покрыл все окрестности белым покрывалом")

array

['Во время обучения модели machine learning, важно обеспечить достаточное количество данных для тренировки',
 'Во время обучения модели машинного обучения, важно обеспечить достаточное количество данных для тренировки',
 'Использование API позволяет интегрировать различные сервисы в ваше приложение, упрощая процесс разработки',
 'The use of an API allows you to integrate various services into your application, simplifying the development process',
 'Для анализа больших объемов данных рекомендуется использовать фреймворк Hadoop, который обеспечивает эффективную обработку данных',
 'В процессе дебаггинга программы была обнаружена ошибка, связанная с неправильным использованием переменных',
 'При разработке user interface необходимо учитывать принципы user experience, чтобы сделать приложение максимально удобным для пользователя',
 'При разработке пользовательского интерфейса необходимо учитывать принципы пользовательского опыта, чтобы сделать приложение максимально удобным для пользовате

## Spacy
python -m spacy download ru_core_news_lg

In [2]:
import time
import torch, torch.nn.functional as F
import pandas as pd
import spacy

# 1. инициализация
start = time.time()
nlp = spacy.load("ru_core_news_lg")        

# 2. батч-обработка
texts = [str(x) for x in array]   # 15 строк
docs  = list(nlp.pipe(texts, batch_size=32))

# 3. тензоры + l2-нормализация
spacy_tensors = torch.stack([
    F.normalize(torch.from_numpy(doc.vector).float(), p=2, dim=0)
    for doc in docs
])

# 4. сохранение
#pd.DataFrame(spacy_vectors.numpy()).to_csv("spacy_vectors.csv", index=False)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(spacy_tensors))
print(type(spacy_tensors[0]))
spacy_tensors         # посмотреть первый вектор

Elapsed: 1.743 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[ 0.0471, -0.0748, -0.0610,  ...,  0.0665,  0.0973,  0.0512],
        [ 0.0189, -0.0891, -0.1064,  ...,  0.0217,  0.1077,  0.0567],
        [ 0.0193,  0.0275,  0.0398,  ...,  0.0553,  0.0604,  0.0291],
        ...,
        [-0.0394,  0.0593,  0.0911,  ...,  0.1502,  0.0110, -0.0526],
        [-0.0327,  0.0338,  0.0763,  ...,  0.1353, -0.0280, -0.0146],
        [-0.0047, -0.1529, -0.0808,  ..., -0.1064, -0.0031,  0.0686]])

## sentence-transformers/paraphrase-multilingual-mpnet-base-v2

In [3]:
from sentence_transformers import SentenceTransformer

start = time.time()

mpnet_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
mpnet_tensors = mpnet_model.encode(array, convert_to_tensor=True, normalize_embeddings=True)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(mpnet_tensors))
print(type(mpnet_tensors[0]))
mpnet_tensors

Elapsed: 3.635 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[-0.0210,  0.0568, -0.0008,  ...,  0.0071, -0.0275, -0.0335],
        [-0.0191,  0.0611, -0.0013,  ...,  0.0122, -0.0316, -0.0372],
        [-0.0119,  0.0471, -0.0041,  ..., -0.0236,  0.0029, -0.0008],
        ...,
        [ 0.0525,  0.0380, -0.0045,  ..., -0.0058, -0.0288, -0.0250],
        [ 0.0138,  0.0647, -0.0034,  ...,  0.0049, -0.0106, -0.0121],
        [-0.0704,  0.0087, -0.0060,  ...,  0.0032, -0.0201, -0.0085]])

## intfloat/multilingual-e5-large-instruct 

In [4]:
from sentence_transformers import SentenceTransformer

custom_array = []
for sentence in array:
    custom_array.append("passage: " + sentence)


print(custom_array[0])

start = time.time()

e5_model = SentenceTransformer("intfloat/multilingual-e5-large-instruct")
e5_tensors = e5_model.encode(custom_array, convert_to_tensor=True, normalize_embeddings=True)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(e5_tensors))
print(type(e5_tensors[0]))
e5_tensors

passage: Во время обучения модели machine learning, важно обеспечить достаточное количество данных для тренировки
Elapsed: 5.641 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[ 0.0224,  0.0028, -0.0332,  ..., -0.0044, -0.0243,  0.0223],
        [ 0.0199,  0.0061, -0.0264,  ..., -0.0029, -0.0290,  0.0193],
        [ 0.0179,  0.0071,  0.0040,  ..., -0.0358, -0.0150,  0.0064],
        ...,
        [ 0.0351,  0.0312, -0.0321,  ...,  0.0072, -0.0395,  0.0002],
        [ 0.0133,  0.0260, -0.0459,  ..., -0.0140, -0.0577,  0.0097],
        [ 0.0262,  0.0196, -0.0255,  ..., -0.0047, -0.0388,  0.0361]])

## DeepPavlov/bert-base-multilingual-cased-sentence
вместо google-bert/bert-base-multilingual-cased

In [5]:
from sentence_transformers import SentenceTransformer

start = time.time()

bert_model = SentenceTransformer("DeepPavlov/bert-base-multilingual-cased-sentence")
bert_tensors = bert_model.encode(array, convert_to_tensor=True, normalize_embeddings=True)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(bert_tensors))
print(type(bert_tensors[0]))
bert_tensors

# from transformers import AutoTokenizer, AutoModel
# import torch

# start = time.time()

# tok = AutoTokenizer.from_pretrained("DeepPavlov/bert-base-multilingual-cased-sentence")
# model = AutoModel.from_pretrained("DeepPavlov/bert-base-multilingual-cased-sentence")

# batch = tok(array, padding=True, truncation=True, return_tensors="pt")

# with torch.no_grad():
#     outputs = model(**batch)        # пример получения hidden-states показан на странице google-bert/bert-base-multilingual-cased

# # mean pooling (как описано для этой модели в metatext-описании)
# attention = batch["attention_mask"].unsqueeze(-1)      # [B, L, 1]
# masked = outputs.last_hidden_state * attention         # обнуляем padded позиции
# bert_tensors = masked.sum(1) / attention.sum(1)          # [B, 768]

# print(f"Elapsed: {time.time() - start:.3f} s")
# print(type(bert_tensors))
# print(type(bert_tensors[0]))
# bert_tensors

No sentence-transformers model found with name DeepPavlov/bert-base-multilingual-cased-sentence. Creating a new one with mean pooling.


Elapsed: 2.743 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[ 0.0263, -0.0470,  0.0655,  ..., -0.0595, -0.0260,  0.0101],
        [ 0.0219, -0.0420,  0.0592,  ..., -0.0443, -0.0290,  0.0142],
        [ 0.0077, -0.0346,  0.0493,  ..., -0.0217, -0.0510, -0.0048],
        ...,
        [ 0.0169,  0.0622,  0.0568,  ...,  0.0151, -0.0283,  0.0099],
        [ 0.0346,  0.0280,  0.0886,  ...,  0.0156,  0.0075, -0.0063],
        [ 0.0196,  0.0048,  0.0900,  ..., -0.0422, -0.0334,  0.0024]])

## ai-forever/FRIDA
вместо ai-forever/sbert_large_nlu_ru 

In [6]:
from sentence_transformers import SentenceTransformer

custom_array = []
for sentence in array:
    custom_array.append("paraphrase: " + sentence)

start = time.time()

frida_model = SentenceTransformer("ai-forever/FRIDA")          
frida_tensors = frida_model.encode(custom_array, convert_to_tensor=True, normalize_embeddings=True)  

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(frida_tensors))
print(type(frida_tensors[0]))
frida_tensors

Elapsed: 12.895 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[-0.0397, -0.0525, -0.0344,  ..., -0.0248,  0.0323,  0.0276],
        [-0.0399, -0.0455, -0.0377,  ..., -0.0258,  0.0314,  0.0259],
        [-0.0326, -0.0403, -0.0391,  ..., -0.0217,  0.0202,  0.0328],
        ...,
        [-0.0448, -0.0081,  0.0346,  ..., -0.0280,  0.0128,  0.0247],
        [-0.0225, -0.0011,  0.0104,  ..., -0.0285,  0.0166,  0.0215],
        [-0.0255, -0.0446, -0.0171,  ..., -0.0453,  0.0269,  0.0007]])

## jinaai/jina-embeddings-v3

In [ ]:
from sentence_transformers import SentenceTransformer

start = time.time()

jina_model = SentenceTransformer("jinaai/jina-embeddings-v3", trust_remote_code=True)

task = "retrieval.passage"
jina_tensors = jina_model.encode(
    array,
    task=task,
    prompt_name=task,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(jina_tensors))
print(type(jina_tensors[0]))
jina_tensors

Elapsed: 51.798 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[ 0.0113, -0.0677,  0.0362,  ...,  0.0254,  0.0039,  0.0319],
        [ 0.0120, -0.0679,  0.0447,  ...,  0.0302,  0.0010,  0.0369],
        [ 0.0285, -0.0402,  0.1276,  ..., -0.0413, -0.0105,  0.0187],
        ...,
        [ 0.0023,  0.0078, -0.0534,  ...,  0.0113,  0.0208, -0.0124],
        [-0.0097, -0.0252, -0.0806,  ...,  0.0033,  0.0004,  0.0085],
        [-0.1331,  0.0163, -0.0874,  ...,  0.0302,  0.0137,  0.0115]])

: 

## Linq-AI-Research/Linq-Embed-Mistral

In [ ]:
from sentence_transformers import SentenceTransformer

start = time.time()

linq_model = SentenceTransformer("Linq-AI-Research/Linq-Embed-Mistral")
linq_tensors = linq_model.encode(array, convert_to_tensor=True, normalize_embeddings=True)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(linq_tensors))
print(type(linq_tensors[0]))
linq_tensors

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]